# grid

> Gridable detection and paged array slicing

In [ ]:
#| default_exp grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import numpy as np, pandas as pd
from paar.grid import is_gridable, array_page

def test_is_gridable():
    assert is_gridable(np.arange(6).reshape(2,3))
    assert is_gridable(np.arange(3))
    assert is_gridable(pd.DataFrame({'a':[1,2]}))
    assert is_gridable(pd.Series([1,2,3]))
    assert not is_gridable(np.zeros((2,2,2)))   # 3-D not gridable
    assert not is_gridable([1,2,3]) and not is_gridable(5)
def test_page_ndarray_2d():
    p = array_page(np.arange(6).reshape(2,3))
    assert p['nrows']==2 and p['ncols']==3
    assert p['headers']==['0','1','2'] and p['index']==['0','1']
    assert p['cells']==[['0','1','2'],['3','4','5']]
def test_page_ndarray_1d():
    p = array_page(np.arange(3))
    assert p['headers']==['0'] and p['index']==['0','1','2'] and p['cells']==[['0'],['1'],['2']]
    assert p['nrows']==3 and p['ncols']==1
def test_page_dataframe():
    p = array_page(pd.DataFrame({'x':[1,2],'y':[3,4]}))
    assert p['headers']==['x','y'] and p['index']==['0','1']
    assert p['cells']==[['1','3'],['2','4']] and p['nrows']==2 and p['ncols']==2
def test_page_series():
    p = array_page(pd.Series([10,20], name='s'))
    assert p['headers']==['s'] and p['cells']==[['10'],['20']] and p['nrows']==2 and p['ncols']==1
def test_page_paging_offsets():
    p = array_page(np.arange(10).reshape(10,1), roff=5, rows=3)
    assert p['index']==['5','6','7'] and p['cells']==[['5'],['6'],['7']] and p['nrows']==10 and p['roff']==5
    assert p['rows']==3
def test_page_non_gridable_none():
    assert array_page([1,2,3]) is None
for t in [test_is_gridable,test_page_ndarray_2d,test_page_ndarray_1d,test_page_dataframe,
          test_page_series,test_page_paging_offsets,test_page_non_gridable_none]: t()

In [ ]:
#| export
from paar.reprs import safe_repr
try: import numpy as np
except Exception: np = None
try: import pandas as pd
except Exception: pd = None

def is_gridable(v):
    "True for a numpy ndarray (1-2D) or a pandas DataFrame/Series."
    if np is not None and isinstance(v, np.ndarray): return v.ndim in (1, 2)
    if pd is not None and isinstance(v, (pd.DataFrame, pd.Series)): return True
    return False

def _cell(x): return safe_repr(x, 40)

def array_page(v, roff=0, coff=0, rows=50, cols=50):
    "Page a gridable value -> dict(headers,index,cells,nrows,ncols,roff,coff,rows,cols), or None."
    if pd is not None and isinstance(v, pd.DataFrame):
        nrows, ncols = v.shape
        sub = v.iloc[roff:roff+rows, coff:coff+cols]
        headers = [str(c) for c in sub.columns]
        index = [str(i) for i in sub.index]
        cells = [[_cell(x) for x in row] for row in sub.values.tolist()]
    elif pd is not None and isinstance(v, pd.Series):
        nrows, ncols = v.shape[0], 1
        sub = v.iloc[roff:roff+rows]
        headers = [str(v.name) if v.name is not None else 'value']
        index = [str(i) for i in sub.index]
        cells = [[_cell(x)] for x in sub.tolist()]
    elif np is not None and isinstance(v, np.ndarray) and v.ndim == 2:
        nrows, ncols = v.shape
        sub = v[roff:roff+rows, coff:coff+cols]
        headers = [str(c) for c in range(coff, min(coff+cols, ncols))]
        index = [str(r) for r in range(roff, min(roff+rows, nrows))]
        cells = [[_cell(x) for x in row] for row in sub.tolist()]
    elif np is not None and isinstance(v, np.ndarray) and v.ndim == 1:
        nrows, ncols = v.shape[0], 1
        sub = v[roff:roff+rows]
        headers = ['0']
        index = [str(r) for r in range(roff, min(roff+rows, nrows))]
        cells = [[_cell(x)] for x in sub.tolist()]
    else:
        return None
    return {'headers': headers, 'index': index, 'cells': cells,
            'nrows': nrows, 'ncols': ncols, 'roff': roff, 'coff': coff,
            'rows': rows, 'cols': cols}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()